In [1]:
# 09_llm_extensions.ipynb -- Peer-review extensions requiring live-model access (P5 contamination, P6 semantic/rarity re-scoring, P7 adversary robustness, P8 multi-language note).
# Reuses data/ artifacts from notebooks 01-02 and the OpenAI config from .env (repo root). Run sections via the RUN_* flags.
# Output paths are intentionally not printed (anonymity).

import os, re, json, hashlib, difflib, urllib.request, urllib.error
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "font.size": 11,
                     "axes.edgecolor": "0.2", "grid.color": "0.85", "image.cmap": "Greys"})
GRAYS = ["0.10", "0.35", "0.60"]; MARKERS = ["o", "s", "^"]

# ---- Which sections to run -------------------------------------------------
RUN_P5 = True     # training-data contamination control (new calls)
RUN_P6 = True     # semantic + rarity-aware re-scoring (reuses cached responses)
RUN_P7 = True     # adversary-robustness sweep (new calls)
# P8 (multi-language) is a config change to notebooks 01-03; see the note at the bottom.

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA, FIG, TAB, CACHE = ROOT/"data", ROOT/"results"/"figures", ROOT/"results"/"tables", ROOT/"data"/"_adv_cache"
for d in (FIG, TAB, CACHE): d.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(FIG/f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG/f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

# ---- OpenAI config (.env at repo root; key never printed) ------------------
def load_env(p):
    cfg = {}
    if p.exists():
        for ln in p.read_text(encoding="utf-8").splitlines():
            ln = ln.strip()
            if ln and not ln.startswith("#") and "=" in ln:
                k, v = ln.split("=", 1); cfg[k.strip()] = v.strip().strip('"').strip("'")
    return cfg
ENV = load_env(ROOT/".env")
API_KEY = ENV.get("OPENAI_API_KEY", os.environ.get("OPENAI_API_KEY", ""))
USE_LLM = bool(API_KEY) and API_KEY.lower() != "none"
MODELS = {"weak": ENV.get("LLM_MODEL_WEAK","gpt-4o-mini"),
          "mid":  ENV.get("LLM_MODEL_MID","gpt-5.4-mini"),
          "strong": ENV.get("LLM_MODEL_STRONG","gpt-5.4")}

def _chat(model, prompt, temperature=0.0, seed=None):
    body = {"model": model, "messages": [{"role":"user","content":prompt}], "temperature": temperature}
    if seed is not None: body["seed"] = seed
    def _post(b):
        req = urllib.request.Request("https://api.openai.com/v1/chat/completions",
                data=json.dumps(b).encode(),
                headers={"Authorization": f"Bearer {API_KEY}", "Content-Type":"application/json"})
        with urllib.request.urlopen(req, timeout=120) as r: return json.loads(r.read())
    try:
        d = _post(body)
    except urllib.error.HTTPError as e:
        msg = e.read().decode(errors="ignore")
        if e.code == 400 and "temperature" in msg:
            body.pop("temperature"); d = _post(body)
        else: raise
    return d["choices"][0]["message"]["content"]

def cached_chat(model, prompt, temperature=0.0, seed=None, tag=""):
    key = hashlib.sha256(f"{model}|{temperature}|{seed}|{tag}|{prompt}".encode()).hexdigest()
    cf = CACHE/f"{key}.json"
    if cf.exists(): return json.loads(cf.read_text(encoding="utf-8"))["content"]
    c = _chat(model, prompt, temperature, seed); cf.write_text(json.dumps({"content":c}), encoding="utf-8"); return c

def parse_map(t):
    i, j = t.find("{"), t.rfind("}")
    if i == -1 or j == -1: return {}
    try: return json.loads(t[i:j+1])
    except Exception: return {}
def norm(s): return str(s).strip().strip("<>").lower()

# ---- Load artifacts from notebooks 01-02 -----------------------------------
corpus = {c["file_id"]: c["code"] for c in json.load(open(DATA/"corpus.json", encoding="utf-8"))}
id_maps = json.load(open(DATA/"id_maps.json", encoding="utf-8"))          # file_id -> {placeholder: original}
masked = json.load(open(DATA/"masked_corpus.json", encoding="utf-8"))     # tier -> [{file_id, code, ...}]
masked_by = {t: {r["file_id"]: r["code"] for r in masked[t]} for t in masked}
SAMPLE = sorted(int(f) for f in id_maps)[:50]
K = 12

def adversary_once(code, idmap, model, temperature=0.0, seed=None, tag=""):
    present = [p for p in idmap if p in code][:K]
    if not present: return None, 0
    prompt = ("You are a security analyst testing whether masked source code leaks its original identifiers. "
              "Placeholders of the form <ID_xxxxxx> replaced identifiers. For each placeholder, infer the MOST LIKELY "
              "original identifier from context. Return ONLY a JSON object mapping each placeholder to one guess.\n"
              f"Placeholders: {present}\n\nCode:\n{code}")
    g = parse_map(cached_chat(model, prompt, temperature, seed, tag))
    correct = sum(1 for p in present if norm(g.get(p,"")) == norm(idmap[p]))
    return correct/len(present), len(present)

# =====================================================================================
# P5 -- TRAINING-DATA CONTAMINATION CONTROL
# =====================================================================================
if RUN_P5 and USE_LLM:
    # (a) Verbatim-completion probe: give the first half of the UNMASKED file, ask the model to continue,
    #     measure line-level overlap with the true continuation. High overlap => memorization of this public repo.
    def verbatim_probe(model, n_files=20, head_lines=12):
        ratios = []
        for fid in SAMPLE[:n_files]:
            lines = corpus[fid].splitlines()
            if len(lines) < head_lines*2: continue
            head = "\n".join(lines[:head_lines]); tail = lines[head_lines:head_lines*2]
            prompt = ("Continue this source file exactly as written. Output only the next lines of code.\n\n"+head)
            out = cached_chat(model, prompt, 0.0, SEED, tag="verbatim").splitlines()
            sm = difflib.SequenceMatcher(None, "\n".join(tail), "\n".join(out[:len(tail)]))
            ratios.append(sm.ratio())
        return float(np.mean(ratios)) if ratios else float("nan")

    # (b) Decontaminated reconstruction: consistently RENAME real identifiers to novel tokens BEFORE masking,
    #     re-mask identifiers, and re-run the adversary scoring against the NOVEL names. If memorization drove
    #     recovery, novel-name recovery should be much lower than original-name recovery.
    JAVA_KW = set("abstract assert boolean break byte case catch char class const continue default do double else enum extends final finally float for goto if implements import instanceof int interface long native new package private protected public return short static strictfp super switch synchronized this throw throws transient try void volatile while true false null var record yield sealed permits".split())
    COMMON = set("String List Map Set Integer Long Double Float Boolean Object Override Exception Collection Optional BigDecimal BigInteger LocalDate LocalDateTime Date Arrays Collections Math System out println get set is to of value name id type data org apache fineract com java util".split())
    def is_code_id(w):
        return (bool(re.search(r"[a-z][A-Z]", w)) or ("_" in w) or any(c.isdigit() for c in w)) and w not in JAVA_KW and w not in COMMON and len(w) >= 3
    def tok(w): return f"<ID_{hashlib.md5(w.encode()).hexdigest()[:6]}>"
    def novel_name(i): return f"Zx{i:04d}qv"   # out-of-distribution identifier unlikely to be memorized

    def decon_file(fid):
        originals = list(dict.fromkeys(id_maps[str(fid)].values()))
        mut = {o: novel_name(i) for i, o in enumerate(originals)}
        code = corpus[fid]
        def repl_mut(m):
            w = m.group(0); return mut.get(w, w)
        mutated = re.sub(r"[A-Za-z_$][A-Za-z0-9_$]*", repl_mut, code)
        idmap = {}
        def repl_mask(m):
            w = m.group(0)
            if w in mut.values() and is_code_id(w):
                t = tok(w); idmap[t] = w; return t
            return w
        masked_code = re.sub(r"[A-Za-z_$][A-Za-z0-9_$]*", repl_mask, mutated)
        return masked_code, idmap

    rows = []
    for level, model in MODELS.items():
        vr = verbatim_probe(model)
        orig = [adversary_once(masked_by["C2"][fid], id_maps[str(fid)], model, tag="p5orig")[0] for fid in SAMPLE]
        orig = [x for x in orig if x is not None]
        dec = []
        for fid in SAMPLE:
            mc, im = decon_file(fid)
            r = adversary_once(mc, im, model, tag="p5decon")[0]
            if r is not None: dec.append(r)
        rows.append({"model": level, "verbatim_overlap": vr,
                     "recovery_original": float(np.mean(orig)) if orig else float("nan"),
                     "recovery_decontaminated": float(np.mean(dec)) if dec else float("nan")})
    p5 = pd.DataFrame(rows); p5.to_csv(TAB/"p5_contamination.csv", index=False)
    print("P5 contamination control:\n", p5.to_string(index=False))

# =====================================================================================
# P6 -- SEMANTIC & RARITY-AWARE RE-SCORING (reuses cached C2 responses)
# =====================================================================================
if RUN_P6:
    freq = {}
    for fid in SAMPLE:
        for o in id_maps[str(fid)].values(): freq[o] = freq.get(o, 0) + 1
    def rarity(o): return "rare" if freq.get(o, 1) <= 2 else "common"
    def semantic_eq(a, b, thr=0.8):
        return difflib.SequenceMatcher(None, norm(a), norm(b)).ratio() >= thr

    if USE_LLM:
        rows = []
        for level, model in MODELS.items():
            for scoring in ["exact", "semantic"]:
                tot = {"rare":0,"common":0}; hit = {"rare":0,"common":0}
                for fid in SAMPLE:
                    code = masked_by["C2"][fid]; idmap = id_maps[str(fid)]
                    present = [p for p in idmap if p in code][:K]
                    if not present: continue
                    prompt = ("You are a security analyst testing whether masked source code leaks its original "
                              "identifiers. Placeholders <ID_xxxxxx> replaced identifiers. For each, infer the most "
                              "likely original identifier. Return ONLY a JSON object mapping each placeholder to one guess.\n"
                              f"Placeholders: {present}\n\nCode:\n{code}")
                    g = parse_map(cached_chat(model, prompt, 0.0, None, tag=""))  # same cache key as nb03 -> hit
                    for p in present:
                        o = idmap[p]; s = rarity(o); tot[s]+=1
                        ok = (norm(g.get(p,""))==norm(o)) if scoring=="exact" else semantic_eq(g.get(p,""), o)
                        hit[s]+= 1 if ok else 0
                for s in ["rare","common"]:
                    rows.append({"model":level,"scoring":scoring,"stratum":s,
                                 "recovery": hit[s]/tot[s] if tot[s] else float("nan"), "n":tot[s]})
        p6 = pd.DataFrame(rows); p6.to_csv(TAB/"p6_semantic_rarity.csv", index=False)
        print("\nP6 semantic/rarity re-scoring:\n", p6.to_string(index=False))

# =====================================================================================
# P7 -- ADVERSARY-ROBUSTNESS SWEEP (k>1 generations; combined file+generation CIs)
# =====================================================================================
if RUN_P7 and USE_LLM:
    KGEN = 5; TEMP = 0.7
    def recov_kgen(model, tier):
        per_file = []
        for fid in SAMPLE:
            code = masked_by[tier][fid]; idmap = id_maps[str(fid)]
            present = [p for p in idmap if p in code][:K]
            if not present: continue
            gs = []
            for g in range(KGEN):
                prompt = ("You are a security analyst. Placeholders <ID_xxxxxx> replaced identifiers in masked code. "
                          "Infer each original identifier. Return ONLY a JSON object of placeholder->guess.\n"
                          f"Placeholders: {present}\n\nCode:\n{code}")
                out = parse_map(cached_chat(model, prompt, TEMP, g, tag=f"kgen{g}"))
                gs.append(sum(1 for p in present if norm(out.get(p,""))==norm(idmap[p]))/len(present))
            per_file.append(gs)
        arr = np.array(per_file)  # files x KGEN
        B = 5000; means = []
        for _ in range(B):
            fi = rng.integers(0, arr.shape[0], arr.shape[0])
            gi = rng.integers(0, arr.shape[1], arr.shape[1])
            means.append(arr[np.ix_(fi, gi)].mean())
        return float(arr.mean()), float(np.percentile(means,2.5)), float(np.percentile(means,97.5))
    rows = []
    for level, model in MODELS.items():
        for tier in ["C2","C3","C4"]:
            m, lo, hi = recov_kgen(model, tier)
            rows.append({"model":level,"tier":tier,"recovery":m,"ci_lo":lo,"ci_hi":hi,"k_gen":KGEN,"temp":TEMP})
    p7 = pd.DataFrame(rows); p7.to_csv(TAB/"p7_adversary_robustness.csv", index=False)
    print("\nP7 adversary robustness (k generations, combined CIs):\n", p7.to_string(index=False))

# =====================================================================================
# P8 -- MULTI-LANGUAGE REPLICATION (configuration change, run separately)
# =====================================================================================
# To replicate on Python instead of Java:
#   1) In notebook 01 set REPO_URL/SUBDIR/COMMIT to a Python OSS project (e.g. a Django or Flask repo),
#      change *.java globbing to *.py, and keep the synthetic-token injection unchanged.
#   2) In notebook 02 replace JAVA_KW with Python keywords and COMMON with common builtins/stdlib names;
#      the camelCase/snake_case/digit identifier heuristic already covers Python (snake_case dominates).
#   3) Re-run notebooks 01-05; compare the paired-test verdicts and the Pareto ordering to the Java run.
# This tests whether the tier ordering is specific to statically typed, verbose Java.

if not USE_LLM:
    print("No API key in .env (OPENAI_API_KEY). P5/P6/P7 that need live calls were skipped; set the key and re-run.")

P5 contamination control:
  model  verbatim_overlap  recovery_original  recovery_decontaminated
  weak          0.579686           0.139208                      0.0
   mid          0.670809           0.338307                      0.0
strong          0.699810           0.643680                      0.0

P6 semantic/rarity re-scoring:
  model  scoring stratum  recovery   n
  weak    exact    rare  0.133144 353
  weak    exact  common  0.156398 211
  weak semantic    rare  0.186969 353
  weak semantic  common  0.165877 211
   mid    exact    rare  0.280453 353
   mid    exact  common  0.407583 211
   mid semantic    rare  0.396601 353
   mid semantic  common  0.426540 211
strong    exact    rare  0.552408 353
strong    exact  common  0.800948 211
strong semantic    rare  0.660057 353
strong semantic  common  0.857820 211

P7 adversary robustness (k generations, combined CIs):
  model tier  recovery    ci_lo    ci_hi  k_gen  temp
  weak   C2  0.155324 0.117811 0.195626      5   0.7
  weak 

In [3]:
# =====================================================================================
# P5B -- MEANING-PRESERVING DECONTAMINATION (fixes the over-correction of novel meaningless ids)
# Rename identifiers by SUB-WORD SYNONYM substitution: the identifier stays semantically plausible and
# guessable from context, but its exact surface string is novel (not the memorized Fineract token). Score
# with near-match, so genuine context-based reconstruction gets credit while memorized exact strings do not.
# =====================================================================================
RUN_P5B = True
if RUN_P5B and USE_LLM:
    THES = {"loan":"credit","account":"ledger","customer":"client","interest":"yield","charge":"fee",
            "savings":"deposit","transaction":"entry","payment":"remittance","validate":"check",
            "process":"handle","create":"make","update":"modify","delete":"remove","get":"fetch",
            "calculate":"compute","compute":"derive","repository":"store","service":"manager",
            "request":"query","response":"result","limit":"cap","rate":"ratio","status":"state",
            "number":"num","amount":"sum","product":"item","schedule":"plan","installment":"tranche",
            "principal":"base","currency":"money","group":"cluster","office":"branch","staff":"agent",
            "report":"summary","config":"setting","exception":"fault","builder":"maker","factory":"creator",
            "mapper":"converter","reader":"loader","writer":"saver","command":"action","event":"signal",
            "entity":"object","field":"attribute","balance":"holding","fund":"pool","period":"span",
            "settlement":"clearing","strategy":"policy","util":"helper","utils":"helpers","date":"day",
            "batch":"bundle","summary":"digest","handler":"processor","fee":"cost","value":"val",
            "strategyutils":"policyhelpers","disbursement":"payout","arrears":"overdue","maturity":"expiry"}
    def split_subwords(w):
        parts = re.split(r"_", w)
        out = []
        for p in parts:
            out += re.findall(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|\d+", p) or [p]
        return [s for s in out if s]
    def recase(src, repl):
        if src.isupper(): return repl.upper()
        if src[:1].isupper(): return repl.capitalize()
        return repl
    def rename_identifier(w):
        subs = split_subwords(w); changed = False; outs = []
        for s in subs:
            key = s.lower()
            if key in THES:
                outs.append(recase(s, THES[key])); changed = True
            else:
                outs.append(s)
        if not changed: return None  # cannot rename meaningfully -> excluded from the estimate
        return ("_" in w and "_".join(outs)) or "".join(outs)
    def semantic_eq(a, b, thr=0.8):
        return difflib.SequenceMatcher(None, norm(a), norm(b)).ratio() >= thr

    def decon_mp_file(fid):
        originals = list(dict.fromkeys(id_maps[str(fid)].values()))
        ren = {o: rename_identifier(o) for o in originals}
        ren = {o: v for o, v in ren.items() if v and v != o}
        if not ren: return None, None
        code = corpus[fid]
        masked_code = re.sub(r"[A-Za-z_$][A-Za-z0-9_$]*",
                             lambda m: f"<ID_{hashlib.md5(ren[m.group(0)].encode()).hexdigest()[:6]}>"
                             if m.group(0) in ren else m.group(0), code)
        idmap = {f"<ID_{hashlib.md5(v.encode()).hexdigest()[:6]}>": v for v in ren.values()}
        return masked_code, idmap

    rows = []
    for level, model in MODELS.items():
        ex, nm = [], []
        for fid in SAMPLE:
            mc, im = decon_mp_file(fid)
            if mc is None: continue
            present = [p for p in im if p in mc][:K]
            if not present: continue
            prompt = ("You are a security analyst. Placeholders <ID_xxxxxx> replaced identifiers in masked code. "
                      "Infer each original identifier from context. Return ONLY a JSON object placeholder->guess.\n"
                      f"Placeholders: {present}\n\nCode:\n{mc}")
            g = parse_map(cached_chat(model, prompt, 0.0, None, tag="p5b"))
            ex.append(sum(1 for p in present if norm(g.get(p,""))==norm(im[p]))/len(present))
            nm.append(sum(1 for p in present if semantic_eq(g.get(p,""), im[p]))/len(present))
        rows.append({"model":level,
                     "recovery_decon_meaning_exact": float(np.mean(ex)) if ex else float("nan"),
                     "recovery_decon_meaning_nearmatch": float(np.mean(nm)) if nm else float("nan")})
    p5b = pd.DataFrame(rows); p5b.to_csv(TAB/"p5b_decon_meaning_preserving.csv", index=False)
    print("\nP5B meaning-preserving decontamination (novel-but-plausible ids):\n", p5b.to_string(index=False))
    # Interpretation: compare to p5_contamination.recovery_original (contaminated, upper bound) and
    # p5_contamination.recovery_decontaminated=0 (meaningless, lower bound). This near-match number is the
    # best in-between estimate of genuine context-based reconstruction on non-memorized code.

# =====================================================================================
# P8 -- MULTI-LANGUAGE REPLICATION (Python) -- self-contained re-run of 01-03 + strong C2-C3 test
# =====================================================================================
RUN_P8 = True
if RUN_P8 and USE_LLM:
    import subprocess, random
    from scipy import stats
    P8_REPO = ENV.get("P8_REPO", "https://github.com/pallets/flask.git")
    P8_SUBDIR = ENV.get("P8_SUBDIR", "src")          # python package dir
    P8_COMMIT = ENV.get("P8_COMMIT", "")              # set a tag/commit for reproducibility; empty = shallow HEAD
    P8_N = int(ENV.get("P8_N", "30"))   # for more statistical power set P8_REPO=django/django, P8_SUBDIR=django in .env
    PSRC = DATA/"_py_repo"
    if not (PSRC/".ok").exists():
        if PSRC.exists():
            import shutil; shutil.rmtree(PSRC)
        def _run(a): subprocess.run(a, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                                    encoding="utf-8", errors="replace")
        if P8_COMMIT:
            _run(["git","clone","--filter=blob:none","--no-checkout",P8_REPO,str(PSRC)])
            _run(["git","-C",str(PSRC),"sparse-checkout","set",P8_SUBDIR])
            _run(["git","-C",str(PSRC),"fetch","--depth","1","origin",P8_COMMIT])
            _run(["git","-C",str(PSRC),"checkout",P8_COMMIT])
        else:
            _run(["git","clone","--depth","1",P8_REPO,str(PSRC)])
        (PSRC/".ok").write_text("ok", encoding="utf-8")
    try:
        HEAD = subprocess.run(["git","-C",str(PSRC),"rev-parse","HEAD"], capture_output=True,
                              text=True).stdout.strip()
    except Exception:
        HEAD = "unknown"

    PY_KW = set("False None True and as assert async await break class continue def del elif else except finally for from global if import in is lambda nonlocal not or pass raise return try while with yield match case".split())
    PY_COMMON = set("self cls str int float bool list dict set tuple bytes object len range print super property staticmethod classmethod Exception ValueError TypeError KeyError return None True False get set add app request response name id type data value key item index result".split())
    def py_is_code_id(w):
        return (("_" in w) or bool(re.search(r"[a-z][A-Z]", w)) or any(c.isdigit() for c in w)) \
               and w not in PY_KW and w not in PY_COMMON and len(w) >= 3

    random.seed(SEED)
    def luhn_full(p):
        def tot(s):
            d=[int(c) for c in s]; t=sum(d[-1::-2])
            for x in d[-2::-2]:
                y=x*2; t+= y-9 if y>9 else y
            return t%10
        return next(p+str(c) for c in range(10) if tot(p+str(c))==0)
    def gen_tokens():
        card="-".join(luhn_full("".join(str(random.randint(0,9)) for _ in range(15)))[i:i+4] for i in range(0,16,4))
        secret="sk-"+"".join(random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789") for _ in range(40))
        email="".join(random.choice("abcdefghijk") for _ in range(6))+"@example.com"
        return {"CARD":card,"SECRET":secret,"EMAIL":email}
    PAT = {"CARD":re.compile(r"\b(?:\d{4}[- ]?){3}\d{4}\b"),
           "SECRET":re.compile(r"\bsk-[A-Za-z0-9]{20,}\b"),
           "EMAIL":re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b")}

    base = PSRC/P8_SUBDIR if (PSRC/P8_SUBDIR).exists() else PSRC
    files = []
    for p in sorted(base.rglob("*.py")):
        try: t = p.read_text(encoding="utf-8", errors="ignore")
        except Exception: continue
        n = t.count("\n")+1
        if 30 <= n <= 500: files.append(t)
    random.shuffle(files); files = files[:P8_N]

    def py_mask(tier, text):
        idmap = {}
        toks = gen_tokens()
        # Build the fixture with % so str.format never runs over the carrier code (which contains literal braces).
        fixture = ("\n\n# fixture (injected)\n_card = \"%s\"\n_secret = \"%s\"\n_email = \"%s\"\n"
                   % (toks["CARD"], toks["SECRET"], toks["EMAIL"]))
        code = text + fixture
        gt = set(toks.values())
        if tier == "C0":
            return code, idmap, gt
        for t, pat in PAT.items():
            code = pat.sub(lambda m: f"<{t}_x>", code)
        if tier in ("C2","C3","C4"):
            KEEP = re.compile(r"(<[A-Z]+_x>)|([A-Za-z_][A-Za-z0-9_]*)")
            def rid(m):
                if m.group(1): return m.group(1)
                w=m.group(2)
                if py_is_code_id(w):
                    k=f"<ID_{hashlib.md5(w.encode()).hexdigest()[:6]}>"; idmap[k]=w; return k
                return w
            code = KEEP.sub(rid, code)
        if tier in ("C3","C4"):
            code = re.sub(r"(<[A-Z_]+_?x?>|\"<STR>\")|(\"[^\"]*\"|'[^']*')", lambda m: m.group(1) or "\"<STR>\"", code)
            code = re.sub(r"(<[A-Z]+_x>)|(\b\d+\.\d+\b)", lambda m: m.group(1) or "<NUM>", code)
        return code, idmap, gt

    def py_recovery(model, tier, masks):
        vals=[]
        for code, idmap in masks[tier]:
            present=[p for p in idmap if p in code][:K]
            if not present: continue
            prompt=("You are a security analyst. Placeholders <ID_xxxxxx> replaced identifiers in masked Python code. "
                    "Infer each original identifier from context. Return ONLY a JSON object placeholder->guess.\n"
                    f"Placeholders: {present}\n\nCode:\n{code}")
            g=parse_map(cached_chat(model, prompt, 0.0, None, tag="p8"))
            vals.append(sum(1 for p in present if norm(g.get(p,""))==norm(idmap[p]))/len(present))
        return vals

    masks = {t: [(lambda r: (r[0], r[1]))(py_mask(t, f)) for f in files] for t in ["C2","C3","C4"]}
    rows=[]; paired={}
    for level, model in MODELS.items():
        per={t: py_recovery(model, t, masks) for t in ["C2","C3","C4"]}
        for t in ["C2","C3","C4"]:
            rows.append({"language":"python","model":level,"tier":t,
                         "recovery": float(np.mean(per[t])) if per[t] else float("nan"), "n":len(per[t])})
        a,b=np.array(per["C2"]),np.array(per["C3"])
        n=min(len(a),len(b)); a,b=a[:n],b[:n]
        if n>0 and not np.allclose(a-b,0):
            w=float(stats.wilcoxon(a,b).pvalue)
        else:
            w=1.0
        paired[level]={"mean_diff_C2_C3":float((a-b).mean()) if n else float("nan"),"wilcoxon_p":w}
    p8=pd.DataFrame(rows); p8.to_csv(TAB/"p8_python_recovery.csv", index=False)
    pd.DataFrame([{"model":k,**v} for k,v in paired.items()]).to_csv(TAB/"p8_python_paired.csv", index=False)
    print(f"\nP8 Python replication (repo HEAD {HEAD[:10]}, N={len(files)}):")
    print(p8.to_string(index=False))
    print("strong C2-C3 paired:", paired.get("strong"))
    print("Compare the strong C2>C3 trade-off and weak/mid equivalence to the Java run to test language-specificity.")

if not USE_LLM:
    print("No API key in .env (OPENAI_API_KEY). Live-call sections (P5/P5B/P6/P7/P8) were skipped; set the key and re-run.")


P5B meaning-preserving decontamination (novel-but-plausible ids):
  model  recovery_decon_meaning_exact  recovery_decon_meaning_nearmatch
  weak                           0.0                          0.072778
   mid                           0.0                          0.131662
strong                           0.0                          0.180066

P8 Python replication (repo HEAD d086db856b, N=15):
language  model tier  recovery  n
  python   weak   C2  0.340741 15
  python   weak   C3  0.251852 15
  python   weak   C4  0.251852 15
  python    mid   C2  0.701852 15
  python    mid   C3  0.572222 15
  python    mid   C4  0.572222 15
  python strong   C2  0.887037 15
  python strong   C3  0.864815 15
  python strong   C4  0.864815 15
strong C2-C3 paired: {'mean_diff_C2_C3': 0.022222222222222223, 'wilcoxon_p': 0.1974660733580187}
Compare the strong C2>C3 trade-off and weak/mid equivalence to the Java run to test language-specificity.
